In [14]:
import torch
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torchvision import transforms
from load_data import load_labelled_and_unlabelled
from sklearn.model_selection import train_test_split
import os, sys
import numpy as np
import random

In [ ]:



class Load_Dataset(Dataset):
    def __init__(self, dataset, dataset_configs):
        super().__init__()
        self.num_channels = dataset_configs.input_channels
        self.return_index = False
        
        # 1. Handle input format: Tuple (from prepare_dataset) or Dict (custom)
        if isinstance(dataset, dict):
            x_data = dataset["samples"]
            y_data = dataset.get("labels")
        elif isinstance(dataset, (tuple, list)):
            x_data = dataset[0]
            y_data = dataset[1] if len(dataset) > 1 else None
        else:
            raise TypeError(f"Dataset must be dict or tuple/list, got {type(dataset)}")

        # 2. Convert to Tensors
        if isinstance(x_data, np.ndarray):
            x_data = torch.from_numpy(x_data)
        
        if y_data is not None and isinstance(y_data, np.ndarray):
            y_data = torch.from_numpy(y_data)
        
        # 3. Fix Data Dimensions (N, C, L)
        if len(x_data.shape) == 2:
            x_data = x_data.unsqueeze(1)
        elif len(x_data.shape) == 3 and x_data.shape[1] != self.num_channels:
            x_data = x_data.permute(0, 2, 1)

        self.transform = None
        self.x_data = x_data.float()
        
        # 4. Efficiently Handle One-Hot Labels
        # Since data_pre_processing.py uses np.eye(), we know it's one-hot.
        if y_data is not None:
            # If (N, C) where C > 1 -> It's one-hot, use argmax
            if y_data.dim() == 2 and y_data.shape[1] > 1:
                y_data = torch.argmax(y_data, dim=1)
            # If (N, 1) -> Just squeeze
            elif y_data.dim() == 2 and y_data.shape[1] == 1:
                y_data = y_data.squeeze(1)
                
            self.y_data = y_data.long()
        else:
            self.y_data = None
            
        self.len = x_data.shape[0]         

    def __getitem__(self, index):
        x = self.x_data[index]
        if self.transform:
            x = self.transform(self.x_data[index].reshape(self.num_channels, -1, 1)).reshape(self.x_data[index].shape)
        y = self.y_data[index] if self.y_data is not None else None
        return x, y

    def __len__(self):
        return self.len

def data_generator(data_path, dataset_configs, hparams, flag, ):
    """
    Load data from .pkl files using load_labelled_and_unlabelled function
    
    Args:
        data_path: Path to the dataset directory
        dataset_configs: Dataset configuration object
        hparams: Hyperparameters dictionary
        flag: 'source' or 'target'
        dtype: 'train', 'val', or 'test'
    """
    # Define activity mapping for harmonization
    activity_mapping = {
        'sitting': 'sitting',
        'Sitting and relaxing': 'sitting',
        'standing': 'standing',
        'Standing still': 'standing',
        'lying': 'lying',
        'Lying down': 'lying',
        'running': 'running',
        'Running': 'running',
        'walking': 'walking',
        'Walking': 'walking',
    }
    
    # Extract dataset name from path
    # dataset_name = os.path.basename(data_path)
    dataset_name = os.path.basename(data_path)
    base_dir = os.path.dirname(data_path)
    pkl_path = os.path.join(base_dir, f"{dataset_name}_processed.pkl")    
    # Load data using load_labelled_and_unlabelled
    prepared_datasets, _, _ = load_labelled_and_unlabelled(
        labelled_dataset_path=pkl_path,
        unlabelled_dataset_path=pkl_path if flag == 'target' else None,  # Use unlabeled for target
        window_size=dataset_configs.sequence_len,
        activity_mapping=activity_mapping,
        verbose=0,
    )
    dataset_files=[]
    # Get the appropriate split based on flag
    if flag == 'source':
            # Source: Train, Test, Validation
            dataset_files.append(prepared_datasets['labelled']['train'])      # Index 0
            dataset_files.append(prepared_datasets['labelled']['test'])       # Index 1
            dataset_files.append(prepared_datasets['labelled']['val']) # Index 2
    else:
        # Target: Unlabelled Train, Unlabelled Test (for adaptation)
        dataset_files.append(prepared_datasets['unlabelled']['train'])
        dataset_files.append(prepared_datasets['labelled']['train'])
    datasets=[Load_Dataset(dataset_files[x], dataset_configs) for x in range(len(dataset_files))]
    # Loading datasets
 
    # if dtype == "test":  # you don't need to shuffle or drop last batch while testing
    #     shuffle = False
    #     drop_last = False
    # else:
    shuffle = dataset_configs.shuffle
    drop_last = dataset_configs.drop_last

    # Dataloaders
    data_loaders=[]
    for dataset in (datasets):
        
        data_loaders.append( torch.utils.data.DataLoader(
            dataset=dataset,
            batch_size=hparams["batch_size"],
            # batch_size=batch_size,
            shuffle=shuffle,
            drop_last=drop_last,
            num_workers=0
        ))

    return data_loaders


In [ ]:
import sys
sys.path.append('..') 
# from dataloader.dataloader import data_generator, few_shot_data_generator
from configs.data_model_configs import get_dataset_class
from configs.hparams import get_hparams_class
import os

def get_configs():
    dataset_class = get_dataset_class('Pamap2')
    hparams_class = get_hparams_class('Pamap2')
    return dataset_class(), hparams_class()
dataset_configs, hparams_class = get_configs()

activity_mapping = {
    'sitting': 'sitting',
    'Sitting and relaxing': 'sitting',
    'standing': 'standing',
    'Standing still': 'standing',
    'lying': 'lying',
    'Lying down': 'lying',
    'running': 'running',
    'Running': 'running',
    'walking': 'walking',
    'Walking': 'walking',
}
data_path='../../ADATIME_data/Pamap2'
flag='target'
    # Extract dataset name from path
# dataset_name = os.path.basename(data_path)
dataset_name = os.path.basename(data_path)
base_dir = os.path.dirname(data_path)
pkl_path = os.path.join(base_dir, f"{dataset_name}_processed.pkl")    
# Load data using load_labelled_and_unlabelled
prepared_datasets, _, _ = load_labelled_and_unlabelled(
    labelled_dataset_path=pkl_path,
    unlabelled_dataset_path=pkl_path if flag == 'target' else None,  # Use unlabeled for target
    window_size=dataset_configs.sequence_len,
    activity_mapping=activity_mapping,
    verbose=0,
)



Unlabeled data shape:  (4510, 200, 18)
Unlabelled Combined shape: (7607, 200, 18)
dict_keys(['labelled', 'unlabelled'])


In [ ]:
print(prepared_datasets['unlabelled'].keys())
print(len((prepared_datasets['labelled']['test'])[0]))

dict_keys(['train', 'val', 'test', 'label_map', 'input_shape', 'output_shape', 'har_users'])


TypeError: object of type 'NoneType' has no len()

In [ ]:
import sys
sys.path.append('..') 
# from dataloader.dataloader import data_generator, few_shot_data_generator
from configs.data_model_configs import get_dataset_class
from configs.hparams import get_hparams_class


def get_configs():
    dataset_class = get_dataset_class('Pamap2')
    hparams_class = get_hparams_class('Pamap2')
    return dataset_class(), hparams_class()
dataset_configs, hparams_class = get_configs()
hparams = {**hparams_class.alg_hparams['DANN'],
                        **hparams_class.train_params}

source_data_path='../../ADATIME_data/Pamap2'
target_data_path='../../ADATIME_data/RealWorld'


dataloaders = data_generator(source_data_path, dataset_configs, hparams,'source')
src_train_dl=dataloaders[0]
src_test_dl=dataloaders[1]
src_val_dl=dataloaders[2]

# self.trg_train_dl = data_generator(self.target_data_path, self.dataset_configs, self.hparams, 'target',"train")
# self.trg_test_dl = data_generator(self.target_data_path, self.dataset_configs, self.hparams, 'target',"test")
dataloaders = data_generator(target_data_path, dataset_configs, hparams, 'target')
trg_train_dl=dataloaders[0]
trg_test_dl=dataloaders[1]

# self.few_shot_dl_5 = few_shot_data_generator(self.trg_test_dl, self.dataset_configs,
#                                              5)  # set 5 to other value if you want other k-shot FST
print("\n--- Dataloader Size Check ---")
print(f"Source Train batches:   {len(src_train_dl)}")
print(f"Target Train batches:   {len(trg_train_dl)}")
print("-------------------------------\n")



Unlabeled data shape:  (24413, 200, 18)
Unlabelled Combined shape: (42058, 200, 18)

--- Dataloader Size Check ---
Source Train batches:   96
Target Train batches:   762
-------------------------------

